## Consumer Financial Protection Bureau (CFPB)

The **CFPB** is a U.S. federal regulatory agency created by the Dodd-Frank Act (2010).
Its mandate is to make markets for consumer financial products and services — mortgages,
credit cards, bank accounts, student & auto loans, debt collection, credit reporting — fair,
transparent, and competitive.

For data purposes, the CFPB is notable because it publishes several **free, public, no-API-key**
datasets and APIs. All CFPB-authored data and source code are, as a work of the U.S. Government,
in the **public domain** within the United States.

- Main data hub: https://www.consumerfinance.gov/data-research/
- Open-source / dev docs: https://cfpb.github.io/
- Primary contact: tech@consumerfinance.gov

## CFPB Data Products — What They Offer & How It's Collected/Reported

| Product | What it is | How it's collected | How it's reported / accessed |
|---|---|---|---|
| **Consumer Complaint Database (CCDB)** | Complaints consumers file about financial products/services. ~Millions of records since 2011. | Consumers submit complaints; CFPB forwards to the company for response. Published after the company responds *or* after 15 days, whichever is first. Complaints about depository institutions <$10B in assets are **not** published. | Public **REST/search API** (JSON, CSV, XLS, XLSX), full-dataset bulk download, and a web search UI. Updates ~daily. |
| **HMDA — Home Mortgage Disclosure Act data** | Loan-level mortgage application & origination records that lenders are legally required to disclose. | Mandatory annual reporting by covered financial institutions (LAR = Loan Application Register). Filed via the HMDA Platform. | Public **Data Browser API**, filtered CSV downloads, aggregate & disclosure reports. Hosted at `ffiec.cfpb.gov`. Published annually. |
| **Mortgage Performance / Delinquency trends** | Mortgage delinquency rates over time and geography. | Derived from servicer/loan-level data via the National Mortgage Database program. | Interactive dashboards + downloadable data on consumerfinance.gov. Updated monthly. |
| **Consumer Credit Trends / market dashboards** | Trends in credit cards, auto loans, mortgages, student loans. | Sampled anonymized credit-bureau panel data (Consumer Credit Panel). | Interactive dashboards, downloadable series. Updated monthly. |
| **Making Ends Meet Survey** | Survey series on consumers' financial status/behavior. | Periodic national consumer surveys run by CFPB's Office of Research. | Downloadable research reports + microdata. |
| **Small Business Lending database (1071)** | Info on small-business credit applications. | Mandatory reporting by covered lenders (rulemaking rollout). | Public database (phased release). |

**Access conventions**
- No API key required for the public data endpoints below.
- CCDB base endpoint: `https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/`
- CCDB API docs (Swagger): https://cfpb.github.io/ccdb5-api/documentation/
- HMDA dev APIs: https://ffiec.cfpb.gov/documentation/category/developer-apis
- Note: The CCDB `frm` pagination offset param is known to be unreliable; prefer date-range or `search_after`/bulk download for large pulls.

In [2]:
import requests
import pandas as pd

CCDB_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

def fetch_complaints(product=None, state=None, date_min=None, date_max=None,
                     search_term=None, size=100):
    """
    Query the CFPB Consumer Complaint Database search API and return a DataFrame.
    All params are optional. No API key required.
    Dates are 'YYYY-MM-DD'.
    """
    params = {
        "size": size,
        "no_aggs": "true",
        # DO NOT pass "format" — that triggers export mode and changes the response shape
    }
    if product:      params["product"] = product
    if state:        params["state"] = state
    if date_min:     params["date_received_min"] = date_min
    if date_max:     params["date_received_max"] = date_max
    if search_term:  params["search_term"] = search_term

    resp = requests.get(
        CCDB_URL,
        params=params,
        headers={"User-Agent": "cfpb-demo/1.0"},
        timeout=60,
    )
    resp.raise_for_status()
    data = resp.json()

    # ---- Resilient parsing: handle both response shapes ----
    hits_field = data.get("hits", [])

    if isinstance(hits_field, dict):
        # Elasticsearch-style: {"hits": {"hits": [...], "total": {...}}}
        records = [hit.get("_source", hit) for hit in hits_field.get("hits", [])]
        total_raw = hits_field.get("total", len(records))
        total = total_raw.get("value", total_raw) if isinstance(total_raw, dict) else total_raw
    elif isinstance(hits_field, list):
        # Export-style: {"hits": [ {complaint}, {complaint}, ... ]}
        records = hits_field
        total = len(records)
    else:
        raise ValueError(f"Unexpected 'hits' type: {type(hits_field)}. Top-level keys: {list(data.keys())}")

    print(f"Matched {total:,} complaints total; returning {len(records)} rows.")
    return pd.DataFrame(records)

df = fetch_complaints(
    product="Checking or savings account",
    date_min="2024-01-01",
    date_max="2024-12-31",
    size=200,
)
df.head()

Matched 52,820 complaints total; returning 200 rows.


,product,complaint_what_happened,date_sent_to_company,issue,sub_product,zip_code,tags,has_narrative,complaint_id,timely,company_response,submitted_via,company,date_received,state,company_public_response,sub_issue
0,Checking or savings account,I had a Checking account with Chase Bank- Some...,2024-09-03T21:43:55.000Z,Managing an account,Checking account,90038,None,True,9999500,Yes,Closed with explanation,Web,JPMORGAN CHASE & CO.,2024-09-03T21:28:03.000Z,CA,None,Deposits and withdrawals
1,Checking or savings account,,2024-09-03T21:56:26.000Z,Managing an account,Checking account,34953,None,False,9999439,Yes,Closed with explanation,Postal mail,JPMORGAN CHASE & CO.,2024-09-03T21:48:18.000Z,FL,None,Problem using a debit or ATM card
2,Checking or savings account,,2024-09-03T23:47:35.000Z,Managing an account,Checking account,95605,None,False,9999411,Yes,Closed with explanation,Web,"SOFI TECHNOLOGIES, INC.",2024-09-03T23:31:35.000Z,CA,None,Problem using a debit or ATM card
3,Checking or savings account,On XXXX XXXX I made a deposit of {$9900.00} wi...,2024-09-04T00:31:52.000Z,Problem caused by your funds being low,Checking account,94544,Older American,True,9999372,Yes,Closed with explanation,Web,JPMORGAN CHASE & CO.,2024-09-03T23:54:57.000Z,CA,None,Bounced checks or returned payments
4,Checking or savings account,I had Axos Bank close out one of my XXXX XXXX ...,2024-09-03T21:47:20.000Z,Managing an account,CD (Certificate of Deposit),43082,Older American,True,9999355,Yes,Closed with explanation,Web,"AXOS FINANCIAL, INC.",2024-09-03T21:08:34.000Z,OH,None,Funds not handled or disbursed as instructed


In [3]:
# A few useful columns the CCDB returns
cols = ["date_received", "product", "sub_product", "issue",
        "company", "state", "company_response", "timely"]
existing = [c for c in cols if c in df.columns]
display(df[existing].head(10))

# Top companies by complaint volume in this slice
print("\nTop 10 companies:")
print(df["company"].value_counts().head(10))

# Complaints per month
if "date_received" in df.columns:
    df["date_received"] = pd.to_datetime(df["date_received"])
    monthly = df.set_index("date_received").resample("ME").size()
    print("\nComplaints per month:")
    print(monthly)

,date_received,product,sub_product,issue,company,state,company_response,timely
0,2024-09-03T21:28:03.000Z,Checking or savings account,Checking account,Managing an account,JPMORGAN CHASE & CO.,CA,Closed with explanation,Yes
1,2024-09-03T21:48:18.000Z,Checking or savings account,Checking account,Managing an account,JPMORGAN CHASE & CO.,FL,Closed with explanation,Yes
2,2024-09-03T23:31:35.000Z,Checking or savings account,Checking account,Managing an account,"SOFI TECHNOLOGIES, INC.",CA,Closed with explanation,Yes
3,2024-09-03T23:54:57.000Z,Checking or savings account,Checking account,Problem caused by your funds being low,JPMORGAN CHASE & CO.,CA,Closed with explanation,Yes
4,2024-09-03T21:08:34.000Z,Checking or savings account,CD (Certificate of Deposit),Managing an account,"AXOS FINANCIAL, INC.",OH,Closed with explanation,Yes
5,2024-09-03T20:26:38.000Z,Checking or savings account,Checking account,Managing an account,JPMORGAN CHASE & CO.,SC,Closed with monetary relief,Yes
6,2024-09-03T22:10:49.000Z,Checking or savings account,CD (Certificate of Deposit),Closing an account,GOLDMAN SACHS BANK USA,OR,Closed with explanation,Yes
7,2024-09-03T23:12:15.000Z,Checking or savings account,Checking account,Managing an account,"BANK OF AMERICA, NATIONAL ASSOCIATION",MO,Closed with explanation,Yes
8,2024-09-03T23:07:18.000Z,Checking or savings account,Checking account,Opening an account,GOLDMAN SACHS BANK USA,OR,Closed with explanation,Yes
9,2024-09-03T11:04:48.000Z,Checking or savings account,Savings account,Managing an account,"AXOS FINANCIAL, INC.",FL,Closed with explanation,Yes



Top 10 companies:
company
WELLS FARGO & COMPANY                                  36
JPMORGAN CHASE & CO.                                   22
BANK OF AMERICA, NATIONAL ASSOCIATION                  22
Chime Financial Inc                                    13
CITIBANK, N.A.                                          8
UNITED SERVICES AUTOMOBILE ASSOCIATION                  6
TRUIST FINANCIAL CORPORATION                            5
SOFI TECHNOLOGIES, INC.                                 5
PNC Bank N.A.                                           5
Fidelity National Information Services, Inc. (FNIS)     5
Name: count, dtype: int64

Complaints per month:
date_received
2024-08-31 00:00:00+00:00     72
2024-09-30 00:00:00+00:00    128
Freq: ME, dtype: int64


In [4]:
import requests
import pandas as pd
from io import StringIO

# HMDA Data Browser API — nationwide loan records, filtered CSV.
# Example: 1st-lien home-purchase originations in PA for 2023.
HMDA_URL = "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv"
params = {
    "years": "2023",
    "states": "PA",
    "actions_taken": "1",   # 1 = loan originated
    "loan_purposes": "1",   # 1 = home purchase
}
r = requests.get(HMDA_URL, params=params,
                 headers={"User-Agent": "cfpb-demo/1.0"}, timeout=120)
r.raise_for_status()
hmda = pd.read_csv(StringIO(r.text), low_memory=False)
print(hmda.shape)
hmda.head()

(116819, 99)


,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-2,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2023,549300VZVN841I2ILS84,37964,PA,42045.0,4.204541e+10,NC,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4930,27.75,83100,214.25,1100,1359,54
1,2023,549300C04BJ0G297NC13,33874,PA,42091.0,4.209121e+10,NC,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4794,21.44,142100,213.04,1091,1079,64
2,2023,B4TYDEB6GKMZO031MB27,37964,PA,42101.0,4.210100e+10,NC,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4501,31.13,83100,250.31,386,378,48
3,2023,549300PL8ER6H23P0Z91,33874,PA,42091.0,4.209120e+10,NC,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,6940,22.71,142100,167.12,1501,1696,31
4,2023,B4TYDEB6GKMZO031MB27,38300,PA,42003.0,4.200356e+10,NC,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,3881,7.47,101900,137.74,1183,1380,48
